# Data processing for Feature Pipeline

In [154]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType
%load_ext autoreload
%autoreload 2
from utils.helper import generate_first_of_month_dates
import utils.data_processing_bronze_table
import utils.data_processing_silver_table

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Set up & config

In [128]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [129]:
# Setup config
start_date_str = "2023-01-01"
label_end_date_str = "2024-12-01"
snapshot_date_str = "2023-01-01"
feature_end_date_str = "2025-01-01"

In [130]:
# Generate date string list
date_str_lst = generate_first_of_month_dates(start_date_str, feature_end_date_str)

## Build bronze tables

In [73]:
# Run bronze backfill
feature_sources = {
    "attributes": "data/features_attributes.csv",
    "financials": "data/features_financials.csv",
    "clickstream": "data/feature_clickstream.csv"
}

for table_name, source_path in feature_sources.items():
    bronze_directory = f"datamart/bronze/{table_name}"
    for date in date_str_lst:
        utils.data_processing_bronze_table.process_feature_bronze_table(
            date,
            source_path,
            table_name,
            bronze_directory,
            spark
        )

2023-01-01 row count: 530
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_01_01.csv
2023-02-01 row count: 501
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_02_01.csv
2023-03-01 row count: 506
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_03_01.csv
2023-04-01 row count: 510
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_04_01.csv
2023-05-01 row count: 521
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_05_01.csv
2023-06-01 row count: 517
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_06_01.csv
2023-07-01 row count: 471
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_07_01.csv
2023-08-01 row count: 481
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_08_01.csv
2023-09-01 row count: 454
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_09_01.csv
2023-10-01 row count: 487
saved to: datamart/bronze/attributes/b

## EDA & Silver preparation for `features_attributes` data

**Goal**: Use EDA to justify the cleaning rules, then build a clean Silver dataframe.

- Keep `customer_id` as the entity/join key;
- Clean `age`;
- Clean `occupation`;
- Keep and validate `snapshot_date` for temporal alignment;
- Exclude `Name` and `SSN` from Silver because they are PII and are not required for downstream feature generation. The raw values remain available in Bronze.


### 1. Exploration at file-level

Confirm that the Bronze partitions have the expected structure and that `Customer_ID` is unique within each snapshot.

In [74]:
# Explore data at file-level
attributes_dir = "datamart/bronze/attributes"
attribute_files = sorted(glob.glob(os.path.join(attributes_dir, "*.csv")))

results = []

for file_path in attribute_files:
    file_name = os.path.basename(file_path)

    # Expected Bronze naming: bronze_attributes_YYYY_MM_DD.csv
    snapshot_date = file_name.replace("bronze_feature_attributes_", "").replace(".csv", "")

    # Read Bronze consistently as strings; type casting will be enforced in Silver
    df = spark.read.csv(file_path, header=True, inferSchema=False)

    row_count = df.count()
    column_count = len(df.columns)
    distinct_customer_count = df.select("customer_id").distinct().count()

    results.append({
        "snapshot_date": snapshot_date,
        "row_number": row_count,
        "column_number": column_count,
        "distinct_customer_id": distinct_customer_count,
        "customer_id_unique_within_snapshot": (
            row_count == distinct_customer_count
        )
    })

attributes_summary = pd.DataFrame(results).sort_values("snapshot_date")
attributes_summary

,snapshot_date,row_number,column_number,distinct_customer_id,customer_id_unique_within_snapshot
0,2023_01_01,530,6,530,True
1,2023_02_01,501,6,501,True
2,2023_03_01,506,6,506,True
3,2023_04_01,510,6,510,True
4,2023_05_01,521,6,521,True
5,2023_06_01,517,6,517,True
6,2023_07_01,471,6,471,True
7,2023_08_01,481,6,481,True
8,2023_09_01,454,6,454,True
9,2023_10_01,487,6,487,True


**Observations**: 6 source columns in every Bronze partition, `Customer_id` is unique in each partition. From here on, all 25 partitions are loaded as one table so each column can be profiled across the full dataset


### 2. Exploration  at union-level

In [75]:
# Union all partitions into one dataframe
attribute_dfs = [
    spark.read.csv(file_path, header=True, inferSchema=False)
    for file_path in attribute_files
]

attributes_df = attribute_dfs[0]
for partition_df in attribute_dfs[1:]:
    attributes_df = attributes_df.unionByName(partition_df)

print(f"Total rows: {attributes_df.count()}")
attributes_df.printSchema()
attributes_df.show(5, truncate=False)

Total rows: 12500
root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: string (nullable = true)

+-----------+---------------+---+-----------+----------+-------------+
|Customer_ID|Name           |Age|SSN        |Occupation|snapshot_date|
+-----------+---------------+---+-----------+----------+-------------+
|CUS_0x1037 |Matthewm       |45 |230-22-9583|Accountant|2023-01-01   |
|CUS_0x1069 |Andreas Cremero|32 |761-27-5143|Accountant|2023-01-01   |
|CUS_0x114a |Valetkevitchu  |43 |133-89-5234|Developer |2023-01-01   |
|CUS_0x1184 |Cohenq         |49 |963-76-2464|Lawyer    |2023-01-01   |
|CUS_0x1297 |Edwardsz       |46 |#F%$D@*&8  |Manager   |2023-01-01   |
+-----------+---------------+---+-----------+----------+-------------+
only showing top 5 rows



#### 2.1. Check duplicates

In [76]:
# Number of rows
total_rows = attributes_df.count()

# Number of exact duplicate rows (excluding the first occurrence)
exact_duplicate_rows = (
    attributes_df.groupBy(attributes_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .select(F.sum(F.col("count") - 1).alias("duplicate_rows"))
    .first()["duplicate_rows"] or 0
)

# Number of duplicated (snapshot_date, customer_id) pairs
duplicated_pairs = (
    attributes_df.groupBy("snapshot_date", "Customer_ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_id = attributes_df.filter(F.col("Customer_ID").isNull()).count()
null_snapshot_date = attributes_df.filter(F.col("snapshot_date").isNull()).count()

print(f"Number of rows: {total_rows}")
print(f"Number of exact duplicated rows: {exact_duplicate_rows}")
print(f"Number of duplicated (snapshot_date, customer_id) pairs: {duplicated_pairs}")
print(f"Null Customer_ID: {null_customer_id}")
print(f"Null snapshot_date: {null_snapshot_date}")

Number of rows: 12500
Number of exact duplicated rows: 0
Number of duplicated (snapshot_date, customer_id) pairs: 0
Null Customer_ID: 0
Null snapshot_date: 0


#### 2.2. Scan generic missing values (null-like strings)

In [77]:
# Values treated as missing, regardless of letter case
missing_values = ("", "unknown", "n/a", "na", "nan", "null", "none")

missing_count_expressions = []

for column_name in attributes_df.columns:
    column_text = F.lower(
        F.trim(F.col(column_name).cast("string"))
    )
    is_missing = (
        F.col(column_name).isNull()
        | column_text.isin(*missing_values)
    )

    missing_count_expressions.append(
        F.sum(F.when(is_missing, 1).otherwise(0)).alias(column_name)
    )

attributes_df.select(*missing_count_expressions).show(truncate=False)

+-----------+----+---+---+----------+-------------+
|Customer_ID|Name|Age|SSN|Occupation|snapshot_date|
+-----------+----+---+---+----------+-------------+
|0          |0   |0  |0  |0         |0            |
+-----------+----+---+---+----------+-------------+



### 3. Clean `Age` column

#### 3.1. Inspect raw data

In [78]:
print("Age source datatype:")
print(attributes_df.schema["Age"])

Age source datatype:
StructField('Age', StringType(), True)


In [57]:
# Check rows that are not numeric
age_invalid = attributes_df.filter(
    ~F.col("Age").rlike(r"^\d+$")
)

age_invalid.select("Age").show(50, truncate=False)
print(f"Invalid Age rows: {age_invalid.count()}")

+-----+
|Age  |
+-----+
|41_  |
|-500 |
|2329_|
|38_  |
|39_  |
|18_  |
|28_  |
|15_  |
|45_  |
|26_  |
|31_  |
|-500 |
|42_  |
|23_  |
|48_  |
|45_  |
|-500 |
|29_  |
|-500 |
|-500 |
|-500 |
|43_  |
|32_  |
|34_  |
|21_  |
|35_  |
|-500 |
|31_  |
|37_  |
|43_  |
|43_  |
|-500 |
|21_  |
|43_  |
|27_  |
|25_  |
|45_  |
|7723_|
|41_  |
|-500 |
|-500 |
|14_  |
|38_  |
|51_  |
|43_  |
|44_  |
|35_  |
|-500 |
|55_  |
|35_  |
+-----+
only showing top 50 rows

Invalid Age rows: 741


**Observations:** `age` contains anomalies:
- Trailing underscores, e.g. `41_`, `2329_`  ==> Formatting issues
- Negative values, e.g: -500  ==> Invalid value

In [81]:
# Values containing an underscore
age_with_underscore = attributes_df.filter(
    F.col("Age").contains("_")
)

# Confirm the underscore pattern is only trailing underscores.
unexpected_underscore_pattern = attributes_df.filter(
    F.col("Age").contains("_")
    & ~F.col("Age").rlike(r"^-?\d+_+$")
)

print(f"Rows containing underscore: {age_with_underscore.count()}")
print(
    "Rows where underscore is NOT a simple trailing-suffix pattern: "
    f"{unexpected_underscore_pattern.count()}"
)

age_with_underscore.select("Age").distinct().orderBy("Age").show(10, truncate=False)

Rows containing underscore: 637
Rows where underscore is NOT a simple trailing-suffix pattern: 0
+-----+
|Age  |
+-----+
|1070_|
|1248_|
|14_  |
|15_  |
|16_  |
|17_  |
|18_  |
|19_  |
|20_  |
|21_  |
+-----+
only showing top 10 rows



#### 3.2. Handle formatting issue & cast to integer

In [85]:
age_profile_df = (
    attributes_df
    # Remove trailing underscore
    .withColumn(
        "age_normalized",
        F.regexp_replace(F.trim(F.col("Age")), r"_+$", "")
    )
    # Cast to integer
    .withColumn(
        "age_parsed",
        F.col("age_normalized").cast(IntegerType())
    )
)

# Verify that the formatting normalisation did not create unparseable ages.
age_profile_df.filter(
    F.col("Age").isNotNull() & F.col("age_parsed").isNull()
).select("Age", "age_normalized").show(50, truncate=False)

print(
    "Rows that could not be parsed after normalisation: "
    f"{age_profile_df.filter(F.col('Age').isNotNull() & F.col('age_parsed').isNull()).count()}"
)


print("Rows after normalisation and parsing:")
age_profile_df.select('Age', 'age_normalized', 'age_parsed').show()

+---+--------------+
|Age|age_normalized|
+---+--------------+
+---+--------------+

Rows that could not be parsed after normalisation: 0
Rows after normalisation and parsing:
+---+--------------+----------+
|Age|age_normalized|age_parsed|
+---+--------------+----------+
| 45|            45|        45|
| 32|            32|        32|
| 43|            43|        43|
| 49|            49|        49|
| 46|            46|        46|
| 35|            35|        35|
| 27|            27|        27|
| 47|            47|        47|
| 15|            15|        15|
|41_|            41|        41|
| 52|            52|        52|
| 19|            19|        19|
| 31|            31|        31|
| 29|            29|        29|
| 36|            36|        36|
| 20|            20|        20|
| 31|            31|        31|
| 27|            27|        27|
| 35|            35|        35|
| 46|            46|        46|
+---+--------------+----------+
only showing top 20 rows



#### 3.3. Inspect range of `Age`

**Business sense:**
- Negative values (e.g: -500) and ages above 100 are implausible ==> Convert to Null
- Values < 18 does not make sense in business & legal context ==> Retain as observed in Silver and flag for Gold 

In [88]:
# Overall range checks
age_counts = age_profile_df.select(
    F.sum(F.when(F.col("age_parsed") < 0, 1).otherwise(0))
        .alias("count_age_below_0"),
    F.sum(F.when(F.col("age_parsed") < 18, 1).otherwise(0))
        .alias("count_age_below_18"),
    F.sum(F.when(F.col("age_parsed") > 100, 1).otherwise(0))
        .alias("count_age_above_100")
)

age_counts.show()


# Inspect frequencies outside the expected 0-100 quality range
(
    age_profile_df
    .filter(
        (F.col("age_parsed") < 0)
        | (F.col("age_parsed") > 100)
        | (F.col("age_parsed") < 18)
    )
    .groupBy("age_parsed")
    .count()
    .orderBy("age_parsed")
    .show(150, truncate=False)
)

+-----------------+------------------+-------------------+
|count_age_below_0|count_age_below_18|count_age_above_100|
+-----------------+------------------+-------------------+
|              104|               773|                215|
+-----------------+------------------+-------------------+

+----------+-----+
|age_parsed|count|
+----------+-----+
|-500      |104  |
|14        |87   |
|15        |195  |
|16        |189  |
|17        |198  |
|169       |1    |
|203       |1    |
|210       |1    |
|236       |1    |
|305       |1    |
|306       |1    |
|325       |1    |
|335       |1    |
|344       |1    |
|395       |1    |
|448       |1    |
|506       |1    |
|509       |1    |
|581       |1    |
|598       |1    |
|655       |2    |
|813       |1    |
|831       |1    |
|886       |1    |
|925       |1    |
|1004      |1    |
|1022      |1    |
|1066      |1    |
|1070      |1    |
|1087      |1    |
|1094      |1    |
|1149      |1    |
|1203      |1    |
|1220      |1    |
|

In [89]:
# Apply the Silver cleaning rule for Age (replace value <0 and >100 with null)
age_clean_df = (
    age_profile_df
    .withColumn(
        "age_clean",
        F.when(
            (F.col("age_parsed") < 0)
            | (F.col("age_parsed") > 100),
            F.lit(None).cast(IntegerType())
        ).otherwise(F.col("age_parsed"))
    )
)

# Validate the rule
remaining_invalid_age = age_clean_df.filter(
    (F.col("age_clean") < 0) | (F.col("age_clean") > 100)
).count()

print(f"Remaining impossible age values: {remaining_invalid_age}")
print(
    "Age values converted to NULL: "
    f"{age_clean_df.filter(F.col('age_clean').isNull()).count()}"
)

Remaining impossible age values: 0
Age values converted to NULL: 319


### 4. Clean `Occupation` column

#### 4.1. Inspect raw data & profile categories

In [63]:
attributes_df.groupBy("Occupation").count().orderBy(F.desc("count")).show()

+-------------+-----+
|   Occupation|count|
+-------------+-----+
|      _______|  880|
|       Lawyer|  828|
|    Architect|  795|
|     Engineer|  793|
|   Accountant|  791|
|    Scientist|  789|
|      Teacher|  782|
|Media_Manager|  780|
|    Developer|  780|
|     Mechanic|  780|
| Entrepreneur|  776|
|   Journalist|  761|
|       Doctor|  760|
|     Musician|  741|
|      Manager|  736|
|       Writer|  728|
+-------------+-----+



#### 4.2. Handle placeholder

Convert placeholder `_______` to Null, retain valid category names as they are

In [93]:
occupation_clean_df = (
    age_clean_df
    .withColumn(
        "occupation_clean",
        F.when(
            F.trim(F.col("Occupation")) == "_______",
            F.lit(None).cast(StringType())
        ).otherwise(F.trim(F.col("Occupation")))
    )
)

print(
    "Occupation NULL count after cleaning: "
    f"{occupation_clean_df.filter(F.col('occupation_clean').isNull()).count()}"
)

(
    occupation_clean_df
    .groupBy("occupation_clean")
    .count()
    .orderBy(F.desc("count"))
    .show(100, truncate=False)
)

Occupation NULL count after cleaning: 880
+----------------+-----+
|occupation_clean|count|
+----------------+-----+
|NULL            |880  |
|Lawyer          |828  |
|Architect       |795  |
|Engineer        |793  |
|Accountant      |791  |
|Scientist       |789  |
|Teacher         |782  |
|Media_Manager   |780  |
|Developer       |780  |
|Mechanic        |780  |
|Entrepreneur    |776  |
|Journalist      |761  |
|Doctor          |760  |
|Musician        |741  |
|Manager         |736  |
|Writer          |728  |
+----------------+-----+



### Build Silver `features_attributes` table

| Source column | Silver decision | Reason |
|---|---|---|
| `Customer_ID` | Keep → `customer_id` | Entity key for joins |
| `Name` | Drop | PII; not required for ML features |
| `Age` | Keep cleaned → `age` | Candidate model feature |
| `SSN` | Drop | Sensitive PII; not required for ML features |
| `Occupation` | Keep cleaned → `occupation` | Candidate model feature |
| `snapshot_date` | Keep as `DateType` | Temporal alignment / leakage control |

In [97]:
silver_attributes_df = (
    occupation_clean_df
    .withColumn(
        "customer_id",
        F.trim(F.col("Customer_ID")).cast(StringType())
    )
    .withColumn(
        "snapshot_date_clean",
        F.to_date(F.col("snapshot_date"), "yyyy-MM-dd")
    )
    .select(
        "customer_id",
        F.col("age_clean").alias("age"),
        F.col("occupation_clean").alias("occupation"),
        F.col("snapshot_date_clean").alias("snapshot_date")
    )
)

silver_attributes_df.printSchema()
silver_attributes_df.show(20, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+-----------+---+-------------+-------------+
|customer_id|age|occupation   |snapshot_date|
+-----------+---+-------------+-------------+
|CUS_0x1037 |45 |Accountant   |2023-01-01   |
|CUS_0x1069 |32 |Accountant   |2023-01-01   |
|CUS_0x114a |43 |Developer    |2023-01-01   |
|CUS_0x1184 |49 |Lawyer       |2023-01-01   |
|CUS_0x1297 |46 |Manager      |2023-01-01   |
|CUS_0x12fb |35 |Doctor       |2023-01-01   |
|CUS_0x1325 |27 |Accountant   |2023-01-01   |
|CUS_0x1341 |47 |Mechanic     |2023-01-01   |
|CUS_0x1375 |15 |Lawyer       |2023-01-01   |
|CUS_0x13a8 |41 |Journalist   |2023-01-01   |
|CUS_0x13ef |52 |Media_Manager|2023-01-01   |
|CUS_0x1440 |19 |Teacher      |2023-01-01   |
|CUS_0x1443 |31 |Lawyer       |2023-01-01   |
|CUS_0x145a |29 |Lawyer       |2023-01-01   |
|CUS_0x1492 |36 |Teacher      |2023-01-01   |
|C

In [100]:
# Validation against Bronze to ensure no rows are accidentally removed

bronze_row_count = attributes_df.count()
silver_row_count = silver_attributes_df.count()

silver_null_summary = silver_attributes_df.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in silver_attributes_df.columns
    ]
)

print(f"Bronze rows: {bronze_row_count}")
print(f"Silver rows: {silver_row_count}")
print(f"Row count preserved: {bronze_row_count == silver_row_count}")
print("Null counts after Silver cleaning:")
silver_null_summary.show(truncate=False)

Bronze rows: 12500
Silver rows: 12500
Row count preserved: True
Null counts after Silver cleaning:
+-----------+---+----------+-------------+
|customer_id|age|occupation|snapshot_date|
+-----------+---+----------+-------------+
|0          |319|880       |0            |
+-----------+---+----------+-------------+



## Build Silver tables for `feature_attributes`

In [7]:
bronze_directory = "datamart/bronze/attributes"
silver_directory = "datamart/silver/attributes"

for snapshot_date in date_str_lst:
    utils.data_processing_silver_table.process_feature_attributes_silver_table(
        snapshot_date,
        bronze_directory,
        silver_directory,
        spark
    )

loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_01_01.csv row count: 530


saved to: datamart/silver/attributes/silver_feature_attributes_2023_01_01.parquet row count: 530
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_02_01.csv row count: 501
saved to: datamart/silver/attributes/silver_feature_attributes_2023_02_01.parquet row count: 501
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_03_01.csv row count: 506
saved to: datamart/silver/attributes/silver_feature_attributes_2023_03_01.parquet row count: 506
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_04_01.csv row count: 510
saved to: datamart/silver/attributes/silver_feature_attributes_2023_04_01.parquet row count: 510
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_05_01.csv row count: 521
saved to: datamart/silver/attributes/silver_feature_attributes_2023_05_01.parquet row count: 521
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_06_01.csv row count: 517
saved to: datamart/silver/attribute

## EDA & Silver Preparation for `feature_financials`

### 1. Exploration at file-leve

In [94]:
# Explore data at file-level
financials_dir = "datamart/bronze/financials"
financials_files = sorted(glob.glob(os.path.join(financials_dir, "*.csv")))

results = []

for file_path in financials_files:
    file_name = os.path.basename(file_path)

    # Expected Bronze naming: bronze_attributes_YYYY_MM_DD.csv
    snapshot_date = file_name.replace("bronze_feature_financials_", "").replace(".csv", "")

    # Read Bronze consistently as strings; type casting will be enforced in Silver
    df = spark.read.csv(file_path, header=True, inferSchema=False)

    row_count = df.count()
    column_count = len(df.columns)
    distinct_customer_count = df.select("customer_id").distinct().count()

    results.append({
        "snapshot_date": snapshot_date,
        "row_number": row_count,
        "column_number": column_count,
        "distinct_customer_id": distinct_customer_count,
        "customer_id_unique_within_snapshot": (
            row_count == distinct_customer_count
        )
    })

financials_summary = pd.DataFrame(results).sort_values("snapshot_date")
financials_summary

,snapshot_date,row_number,column_number,distinct_customer_id,customer_id_unique_within_snapshot
0,2023_01_01,530,22,530,True
1,2023_02_01,501,22,501,True
2,2023_03_01,506,22,506,True
3,2023_04_01,510,22,510,True
4,2023_05_01,521,22,521,True
5,2023_06_01,517,22,517,True
6,2023_07_01,471,22,471,True
7,2023_08_01,481,22,481,True
8,2023_09_01,454,22,454,True
9,2023_10_01,487,22,487,True


### 2. Exploration at union-level

In [ ]:
# Union all partitions into one dataframe
financials_dfs = [
    spark.read.csv(file_path, header=True, inferSchema=False)
    for file_path in financials_files
]

financials_df = financials_dfs[0]
for partition_df in financials_dfs[1:]:
    financials_df = financials_df.unionByName(partition_df)

print(f"Total rows: {financials_df.count()}")
financials_df.printSchema()
financials_df.show(5, truncate=False)

Total rows: 12500
root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: string (nullable = true)
 |-- Num_Bank_Accounts: string (nullable = true)
 |-- Num_Credit_Card: string (nullable = true)
 |-- Interest_Rate: string (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: string (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: string (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: string (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: string (nullable = true)
 |-- Amount_invested_monthly: string (nullable = true)
 |-- Payment_Behaviour: string (nullable = t

#### 2.1. Check duplicates

In [96]:
# Number of exact duplicate rows (excluding the first occurrence)
exact_duplicate_rows = (
    financials_df.groupBy(financials_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .select(F.sum(F.col("count") - 1).alias("duplicate_rows"))
    .first()["duplicate_rows"] or 0
)

# Number of duplicated (snapshot_date, customer_id) pairs
duplicated_pairs = (
    financials_df.groupBy("snapshot_date", "Customer_ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_id = financials_df.filter(F.col("Customer_ID").isNull()).count()
null_snapshot_date = financials_df.filter(F.col("snapshot_date").isNull()).count()

print(f"Number of exact duplicated rows: {exact_duplicate_rows}")
print(f"Number of duplicated (snapshot_date, customer_id) pairs: {duplicated_pairs}")
print(f"Null Customer_ID: {null_customer_id}")
print(f"Null snapshot_date: {null_snapshot_date}")

Number of exact duplicated rows: 0
Number of duplicated (snapshot_date, customer_id) pairs: 0
Null Customer_ID: 0
Null snapshot_date: 0


#### 2.2. Scan generic missing values

In [97]:
# Values treated as missing, regardless of letter case
missing_values = ("", "unknown", "n/a", "na", "nan", "null", "none")

missing_count_expressions = []

for column_name in financials_df.columns:
    column_text = F.lower(
        F.trim(F.col(column_name).cast("string"))
    )
    is_missing = (
        F.col(column_name).isNull()
        | column_text.isin(*missing_values)
    )

    missing_count_expressions.append(
        F.sum(F.when(is_missing, 1).otherwise(0)).alias(column_name)
    )

financials_df.select(*missing_count_expressions).show(truncate=False)

+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+------------------+---------------------+-------------------+-----------------------+-----------------+---------------+-------------+
|Customer_ID|Annual_Income|Monthly_Inhand_Salary|Num_Bank_Accounts|Num_Credit_Card|Interest_Rate|Num_of_Loan|Type_of_Loan|Delay_from_due_date|Num_of_Delayed_Payment|Changed_Credit_Limit|Num_Credit_Inquiries|Credit_Mix|Outstanding_Debt|Credit_Utilization_Ratio|Credit_History_Age|Payment_of_Min_Amount|Total_EMI_per_month|Amount_invested_monthly|Payment_Behaviour|Monthly_Balance|snapshot_date|
+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+-----

#### 2.3. Inspect numeric columns

In [98]:
# Numeric columns stored as text in Bronze, with their target Silver type
numeric_columns = {
    "Annual_Income": FloatType(),
    "Monthly_Inhand_Salary": FloatType(),
    "Num_Bank_Accounts": IntegerType(),
    "Num_Credit_Card": IntegerType(),
    "Interest_Rate": IntegerType(),
    "Num_of_Loan": IntegerType(),
    "Delay_from_due_date": IntegerType(),
    "Num_of_Delayed_Payment": IntegerType(),
    "Changed_Credit_Limit": FloatType(),
    "Num_Credit_Inquiries": IntegerType(),
    "Outstanding_Debt": FloatType(),
    "Credit_Utilization_Ratio": FloatType(),
    "Total_EMI_per_month": FloatType(),
    "Amount_invested_monthly": FloatType(),
    "Monthly_Balance": FloatType(),
}

numeric_pattern = r"^-?\d+(\.\d+)?$"

invalid_numeric_results = []

for column_name in numeric_columns:
    invalid_df = financials_df.filter(~F.trim(F.col(column_name)).rlike(numeric_pattern))
    examples = [
        row[column_name]
        for row in invalid_df.groupBy(column_name).count()
                             .orderBy(F.desc("count")).limit(5).collect()
    ]
    invalid_numeric_results.append({
        "column": column_name,
        "non_numeric_rows": invalid_df.count(),
        "top_examples": examples
    })

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(invalid_numeric_results)

,column,non_numeric_rows,top_examples
0,Annual_Income,859,"[58511.78_, 88120.12_, 57738.06_, 17290.42_, 63847.8_]"
1,Monthly_Inhand_Salary,0,[]
2,Num_Bank_Accounts,0,[]
3,Num_Credit_Card,0,[]
4,Interest_Rate,0,[]
5,Num_of_Loan,623,"[2_, 4_, 3_, 0_, 1_]"
6,Delay_from_due_date,0,[]
7,Num_of_Delayed_Payment,374,"[18_, 19_, 20_, 15_, 12_]"
8,Changed_Credit_Limit,254,[_]
9,Num_Credit_Inquiries,0,[]


**Observations:** 6 numeric columns contain non-numeric text, following 3 patterns:
- Trailing underscores, e.g: `52312.68_`, `2_`  ==> Formatting issues (`Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, `Outstanding_Debt`)
- 1 underscore only `_`  ==> Can be placeholder for missing (`Changed_Credit_Limit`)
- Underscores before and after digits, e.g: `__10000__`, `__-333333333333333333333333333__`  ==> Sentinel values (`Amount_invested_monthly`, `Monthly_Balance`)

In [99]:
# Confirm that every non-numeric value falls into one of the 3 patterns
pattern_results = []

for column_name in numeric_columns:
    raw = F.trim(F.col(column_name))
    pattern_results.append(
        financials_df.select(
            F.lit(column_name).alias("column"),
            F.sum(F.when(raw.rlike(r"^-?\d+(\.\d+)?_+$"), 1).otherwise(0)).alias("trailing_underscore"),
            F.sum(F.when(raw.rlike(r"^_+$"), 1).otherwise(0)).alias("lone_underscore"),
            F.sum(F.when(raw.rlike(r"^__.*__$"), 1).otherwise(0)).alias("wrapped_sentinel"),
            F.sum(F.when(
                ~raw.rlike(numeric_pattern)
                & ~raw.rlike(r"^-?\d+(\.\d+)?_+$")
                & ~raw.rlike(r"^_+$")
                & ~raw.rlike(r"^__.*__$"), 1).otherwise(0)).alias("unexpected_pattern")
        ).toPandas()
    )

pattern_summary = pd.concat(pattern_results, ignore_index=True)
pattern_summary[pattern_summary.drop(columns="column").sum(axis=1) > 0]

,column,trailing_underscore,lone_underscore,wrapped_sentinel,unexpected_pattern
0,Annual_Income,859,0,0,0
5,Num_of_Loan,623,0,0,0
7,Num_of_Delayed_Payment,374,0,0,0
8,Changed_Credit_Limit,0,254,0,0
10,Outstanding_Debt,139,0,0,0
13,Amount_invested_monthly,0,0,558,0
14,Monthly_Balance,0,0,1,0


### 3. Cleaning

#### 3.1. Parse numeric columns

**Cleaning rules:**
- Trailing underscore ==> Remove underscore, keep value
- Lone underscore / wrapped sentinel ==> Convert to Null (the value carries no information)

In [100]:
def parse_numeric(column_name, target_type):
    raw = F.trim(F.col(column_name))
    is_sentinel = raw.rlike(r"^_+$") | raw.rlike(r"^__.*__$")
    normalized = F.regexp_replace(raw, r"_+$", "")
    
    # Cast via double first so integer columns stored as "3.0" parse correctly
    return F.when(is_sentinel, F.lit(None)).otherwise(normalized).cast("double").cast(target_type)


financials_parsed_df = financials_df
for column_name, target_type in numeric_columns.items():
    financials_parsed_df = financials_parsed_df.withColumn(
        column_name, parse_numeric(column_name, target_type)
    )

# Verify: nulls after parsing should only come from sentinels
parse_check = []
for column_name in numeric_columns:
    raw = F.trim(financials_df[column_name])
    parse_check.append({
        "column": column_name,
        "sentinel_rows": financials_df.filter(raw.rlike(r"^_+$") | raw.rlike(r"^__.*__$")).count(),
        "null_after_parsing": financials_parsed_df.filter(F.col(column_name).isNull()).count()
    })

parse_check_df = pd.DataFrame(parse_check)
parse_check_df["unexpected_nulls"] = parse_check_df["null_after_parsing"] - parse_check_df["sentinel_rows"]
parse_check_df

,column,sentinel_rows,null_after_parsing,unexpected_nulls
0,Annual_Income,0,0,0
1,Monthly_Inhand_Salary,0,0,0
2,Num_Bank_Accounts,0,0,0
3,Num_Credit_Card,0,0,0
4,Interest_Rate,0,0,0
5,Num_of_Loan,0,0,0
6,Delay_from_due_date,0,0,0
7,Num_of_Delayed_Payment,0,0,0
8,Changed_Credit_Limit,254,254,0
9,Num_Credit_Inquiries,0,0,0


#### 3.2. Range check 

##### 3.2.1. Distribution

In [101]:
count_columns = [
    "Num_Bank_Accounts", 
    "Num_Credit_Card", 
    "Interest_Rate", 
    "Num_of_Loan",
    "Delay_from_due_date", 
    "Num_of_Delayed_Payment", 
    "Num_Credit_Inquiries",
    "Changed_Credit_Limit"
]

distribution_rows = []
for column_name in count_columns:
    stats = financials_parsed_df.select(
        F.min(column_name).alias("min"),
        F.percentile_approx(column_name, 0.5).alias("p50"),
        F.percentile_approx(column_name, 0.95).alias("p95"),
        F.percentile_approx(column_name, 0.99).alias("p99"),
        F.max(column_name).alias("max"),
        F.sum(F.when(F.col(column_name) < 0, 1).otherwise(0)).alias("negative_rows")
    ).first().asDict()
    distribution_rows.append({"column": column_name, **stats})

pd.DataFrame(distribution_rows)

,column,min,p50,p95,p99,max,negative_rows
0,Num_Bank_Accounts,-1.00,6.00,10.0,511.000000,1756.000000,4
1,Num_Credit_Card,0.00,5.00,10.0,874.000000,1499.000000,0
2,Interest_Rate,1.00,14.00,33.0,2865.000000,5789.000000,0
3,Num_of_Loan,-100.00,3.00,8.0,9.000000,1495.000000,500
4,Delay_from_due_date,-5.00,18.00,54.0,61.000000,67.000000,88
5,Num_of_Delayed_Payment,-3.00,14.00,24.0,27.000000,4293.000000,84
6,Num_Credit_Inquiries,0.00,6.00,14.0,884.000000,2554.000000,0
7,Changed_Credit_Limit,-6.49,9.41,23.6,28.790001,36.970001,204


In [102]:
money_columns = [
    "Annual_Income", 
    "Monthly_Inhand_Salary", 
    "Outstanding_Debt", 
    "Total_EMI_per_month",
    "Amount_invested_monthly", 
    "Monthly_Balance", 
]

distribution2_rows = []
for column_name in money_columns:
    stats = financials_parsed_df.select(
        F.min(column_name).alias("min"),
        F.percentile_approx(column_name, 0.5).alias("p50"),
        F.percentile_approx(column_name, 0.95).alias("p95"),
        F.percentile_approx(column_name, 0.99).alias("p99"),
        F.max(column_name).alias("max"),
        F.sum(F.when(F.col(column_name) < 0, 1).otherwise(0)).alias("negative_rows")
    ).first().asDict()
    distribution2_rows.append({"column": column_name, **stats})

pd.DataFrame(distribution2_rows)

,column,min,p50,p95,p99,max,negative_rows
0,Annual_Income,7005.930176,37551.781250,134282.843750,178793.921875,2.383470e+07,0
1,Monthly_Inhand_Salary,303.645416,3085.853271,10803.636719,13788.976562,1.520463e+04,0
2,Outstanding_Debt,0.230000,1166.079956,4073.620117,4804.870117,4.998070e+03,0
3,Total_EMI_per_month,0.000000,72.876320,590.557556,57478.000000,8.197100e+04,0
4,Amount_invested_monthly,0.000000,127.332077,613.877808,1015.943848,1.977326e+03,0
5,Monthly_Balance,0.382558,339.728455,863.627319,1126.623535,1.463792e+03,0


**Observations:***
- `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, `Num_of_Loan`, `Num_of_Delayed_Payment`, `Num_Credit_Inquiries`: p95 is small but max is in the hundreds/thousands
- Negative values in `Num_Bank_Accounts`, `Num_of_Loan` (-100) and `Num_of_Delayed_Payment` ==> A count cannot be negative

##### 3.2.2. Find natural breakpoint for count columns 

In [103]:
from pyspark.sql.window import Window


def inspect_value_gaps(df, column):
    w = Window.orderBy(column)

    return (
        df
        .select(column)
        .where(F.col(column).isNotNull())
        .groupBy(column)
        .count()
        .withColumn(
            "previous_value",
            F.lag(column).over(w)
        )
        .withColumn(
            "gap",
            F.col(column) - F.col("previous_value")
        )
        .orderBy(F.col(column).asc())
    )

In [104]:
gap_columns = [
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Num_Credit_Inquiries",
    "Changed_Credit_Limit",
]

for column_name in gap_columns:
    print(f"Value gaps for {column_name}:")
    (
        inspect_value_gaps(financials_parsed_df, column_name)
        .filter(F.col("gap") > 1)
        .show(50, truncate=False)
    )

Value gaps for Num_Bank_Accounts:
+-----------------+-----+--------------+---+
|Num_Bank_Accounts|count|previous_value|gap|
+-----------------+-----+--------------+---+
|26               |1    |11            |15 |
|28               |1    |26            |2  |
|50               |1    |28            |22 |
|53               |1    |50            |3  |
|57               |1    |53            |4  |
|64               |1    |57            |7  |
|82               |1    |64            |18 |
|122              |1    |82            |40 |
|129              |1    |122           |7  |
|160              |1    |129           |31 |
|179              |1    |162           |17 |
|182              |1    |179           |3  |
|194              |1    |182           |12 |
|196              |1    |194           |2  |
|203              |1    |196           |7  |
|210              |1    |203           |7  |
|216              |1    |210           |6  |
|224              |1    |216           |8  |
|229              |2 

+--------------------+-----+--------------+---------+
|Changed_Credit_Limit|count|previous_value|gap      |
+--------------------+-----+--------------+---------+
|36.97               |1    |34.85         |2.1200027|
+--------------------+-----+--------------+---------+



**Observations**: Every count column has a dense cluster of realistic values followed by a large jump

E.g: A dense and continuous Interest Rate range of 1–34. The next observed value was 99, after which values were highly sparse, typically occurring only once. Values above 34 were therefore treated as data anomalies 

==> **Cleaning rule**: Thresholds are set as round numbers slightly above the observed cluster, so that realistic future values are not nulled by accident. Values outside the range ==> Convert to Null.

| Column | Valid range in Silver | Reason |
|---|---|---|
| `Num_Bank_Accounts` | 0 – 11 | Cluster ends at 11, next value 26; negative count impossible |
| `Num_Credit_Card` | 0 – 11 | Cluster ends at 11, next value 21 |
| `Interest_Rate` | 0 – 34 (%) | Cluster ends at 34, next value 99 |
| `Num_of_Delayed_Payment` | 0 – 28 | Cluster ends at 28, next value 47; negative count impossible |
| `Num_Credit_Inquiries` | 0 – 17 | Cluster ends at 17, next value 26 |
| `Num_of_Loan` |0-9 | Can be recovered from `Type_of_Loan` |
| `Delay_from_due_date` | Keep as observed | Negative is plausible, in a continuous range |
| `Changed_Credit_Limit` | Keep as observed | Negative is plausible, in a continuous range |

#### 3.3. String columns

In [105]:
string_cols = [
    'Customer_ID',
    'Type_of_Loan',
    'Credit_Mix',
    'Credit_History_Age',
    'Payment_of_Min_Amount',
    'Payment_Behaviour'
]

In [106]:
# Check null rows and number of distinct values for string columns
summary_exprs = []
for column_name in string_cols:
    summary_exprs.extend([
        F.sum(F.col(column_name).isNull().cast("int")).alias(
            f"{column_name}_null_rows"
        ),
        F.countDistinct(column_name).alias(
            f"{column_name}_distinct_values"
        ),
    ])

summary = financials_parsed_df.agg(*summary_exprs).first().asDict()

string_profile = pd.DataFrame([
    {
        "string_column": column_name,
        "null_rows": summary[f"{column_name}_null_rows"],
        "distinct_values": summary[f"{column_name}_distinct_values"],
    }
    for column_name in string_cols
])

string_profile

,string_column,null_rows,distinct_values
0,Customer_ID,0,12500
1,Type_of_Loan,1426,6260
2,Credit_Mix,0,4
3,Credit_History_Age,0,393
4,Payment_of_Min_Amount,0,3
5,Payment_Behaviour,0,7


**Observation:** 
- `Credit_Mix`, `Payment_of_Min_Amount`, `Payment_Behavior` has small distinct values ==> Categorical 
- `Type_of_Loan`: Missing 1426 rows ==> Cross-check with `Num_of_loan`

##### 3.3.1. Handle placeholders for Categorical columns 

In [107]:
categorical_columns = [
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour"
]

for column_name in categorical_columns:
    print(f"Value counts for {column_name}:")
    (
        financials_parsed_df
        .groupBy(column_name)
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

Value counts for Credit_Mix:
+----------+-----+
|Credit_Mix|count|
+----------+-----+
|Standard  |4497 |
|Good      |3032 |
|_         |2611 |
|Bad       |2360 |
+----------+-----+

Value counts for Payment_of_Min_Amount:
+---------------------+-----+
|Payment_of_Min_Amount|count|
+---------------------+-----+
|Yes                  |6571 |
|No                   |4491 |
|NM                   |1438 |
+---------------------+-----+

Value counts for Payment_Behaviour:
+--------------------------------+-----+
|Payment_Behaviour               |count|
+--------------------------------+-----+
|Low_spent_Small_value_payments  |3202 |
|High_spent_Medium_value_payments|2242 |
|Low_spent_Medium_value_payments |1686 |
|High_spent_Large_value_payments |1683 |
|High_spent_Small_value_payments |1389 |
|Low_spent_Large_value_payments  |1300 |
|!@9#%8                          |998  |
+--------------------------------+-----+



**Cleaning rules:**
- `Credit_Mix`: Convert __ to Null
- `Payment_of_Min_Amount`: Convert NM to Null
- `Payment_Behaviour`: Convert !@9#%8 to Null

In [108]:
categorical_placeholders = {
    "Credit_Mix": ["_"],
    "Payment_of_Min_Amount": ["NM"],
    "Payment_Behaviour": ["!@9#%8"],
}

categorical_clean_df = financials_parsed_df
for column_name, placeholders in categorical_placeholders.items():
    categorical_clean_df = categorical_clean_df.withColumn(
        column_name,
        F.when(
            F.trim(F.col(column_name)).isin(*placeholders),
            F.lit(None).cast(StringType())
        ).otherwise(F.trim(F.col(column_name)))
    )

for column_name in categorical_placeholders:
    print(
        f"{column_name} NULL count after cleaning: "
        f"{categorical_clean_df.filter(F.col(column_name).isNull()).count()}"
    )
    categorical_clean_df.groupBy(column_name).count().orderBy(F.desc("count")).show(truncate=False)

Credit_Mix NULL count after cleaning: 2611
+----------+-----+
|Credit_Mix|count|
+----------+-----+
|Standard  |4497 |
|Good      |3032 |
|NULL      |2611 |
|Bad       |2360 |
+----------+-----+

Payment_of_Min_Amount NULL count after cleaning: 1438
+---------------------+-----+
|Payment_of_Min_Amount|count|
+---------------------+-----+
|Yes                  |6571 |
|No                   |4491 |
|NULL                 |1438 |
+---------------------+-----+

Payment_Behaviour NULL count after cleaning: 998
+--------------------------------+-----+
|Payment_Behaviour               |count|
+--------------------------------+-----+
|Low_spent_Small_value_payments  |3202 |
|High_spent_Medium_value_payments|2242 |
|Low_spent_Medium_value_payments |1686 |
|High_spent_Large_value_payments |1683 |
|High_spent_Small_value_payments |1389 |
|Low_spent_Large_value_payments  |1300 |
|NULL                            |998  |
+--------------------------------+-----+



#### 3.4. Cross-columns check between `Type_of_Loan` and `Num_of_loans`

In [109]:
# Normalize values of Type_of_Loan (", and" into ","), then count the number of loan type in row
loan_check_df = (
    categorical_clean_df
    .withColumn(
        "loan_type_count",
        F.when(
            F.col("Type_of_Loan").isNull(),
            F.lit(0)
        ).otherwise(
            F.size(
                F.split(
                    F.regexp_replace(
                        F.col("Type_of_Loan"),
                        r", and ",
                        ", "
                    ),
                    r",\s*"
                )
            )
        )
    )
)

In [110]:
# Find the mismatch between Numb_of_loans and newly created `loan_type_count`

loan_mismatch_df = loan_check_df.filter(
    ~F.col("Num_of_Loan").eqNullSafe(F.col("loan_type_count"))
)

mismatch_row_count = loan_mismatch_df.count()
print(f"Number of mismatching rows: {mismatch_row_count}")

loan_mismatch_df.select(
    "Type_of_Loan",
    "Num_of_Loan",
    "loan_type_count"
).show(mismatch_row_count, truncate=False)

Number of mismatching rows: 567
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+
|Type_of_Loan                                                                                                                                                                |Num_of_Loan|loan_type_count|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+
|Personal Loan, Auto Loan, and Not Specified                                                                                                                                 |119        |3              |
|Personal Loan, and Not Specified                                                                                                                           

**Cleaning rules**
- If `Num_of_Loan` is within reasonable range (0, 9) and match with `loan_type_count` → retain.
- If `Num_of_Loan` falls out of range (0, 9) but `loan_type_count` has data → Use `loan_type_count` data
- If `Type_of_Loan` = NULL and `Num_of_Loan` = 0 → Retain 0.
- If `Type_of_Loan` = NULL but `Num_of_Loan` falls out of range (0, 9) → Null
- If `Type_of_Loan` = NULL but `Num_of_Loan` within range (0, 9) → retain `Num_of_Loan`

In [111]:
num_of_loan = F.col("Num_of_Loan")
loan_type_count = F.col("loan_type_count")
type_of_loan = F.col("Type_of_Loan")

loan_clean_df = loan_check_df.withColumn(
    "Num_of_Loan_clean",
    # Type_of_Loan is NULL
    F.when(
        type_of_loan.isNull() & num_of_loan.between(0, 9),
        num_of_loan
    )

    # Num_of_Loan is valid and matches loan_type_count
    .when(
        type_of_loan.isNotNull()
        & num_of_loan.between(0, 9)
        & (num_of_loan == loan_type_count),
        num_of_loan
    )

    # Num_of_Loan is out of range but loan_type_count is available
    .when(
        type_of_loan.isNotNull()
        & ~num_of_loan.between(0, 9)
        & loan_type_count.isNotNull(),
        loan_type_count
    )
    # All remaining inconsistent / unusable cases
    .otherwise(
        F.lit(None).cast(IntegerType())
    )
)

#### 3.5. Cross-column check between `Annual_Income` and `Monthly_Inhand_Salary`

In [112]:
income_check_df = loan_clean_df.withColumn(
    "annual_income_to_salary_ratio",
    F.when(
        F.col("Monthly_Inhand_Salary") > 0,
        F.col("Annual_Income") /
        (F.col("Monthly_Inhand_Salary") * 12)
    )
)

income_check_df.select(
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "annual_income_to_salary_ratio"
).summary(
    "min", "25%", "50%", "75%", "95%", "99%", "max"
).show()

+-------+-------------+---------------------+-----------------------------+
|summary|Annual_Income|Monthly_Inhand_Salary|annual_income_to_salary_ratio|
+-------+-------------+---------------------+-----------------------------+
|    min|      7005.93|            303.64542|           0.6741517258186552|
|    25%|     19453.05|            1624.7517|           0.9641223231228975|
|    50%|     37551.78|            3085.8533|           1.0034731712893914|
|    75%|     72690.14|             5944.243|           1.0490699620753805|
|    95%|    134282.84|            10803.637|           1.2174283443600677|
|    99%|    178793.92|            13788.977|           1.7009618557979378|
|    max|  2.3834698E7|            15204.634|            3859.589431259365|
+-------+-------------+---------------------+-----------------------------+



In [113]:
income_check_df.filter(
    F.col("annual_income_to_salary_ratio") > 1.5
).select(
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "annual_income_to_salary_ratio"
).orderBy(
    "Annual_Income"
).show(120, truncate=False)

+-------------+---------------------+-----------------------------+
|Annual_Income|Monthly_Inhand_Salary|annual_income_to_salary_ratio|
+-------------+---------------------+-----------------------------+
|7039.745     |303.64542            |1.9320081100022914           |
|7046.5       |355.20834            |1.6531378299120234           |
|7080.7       |358.05832            |1.647939273886287            |
|7087.24      |361.60333            |1.633290588115089            |
|7189.54      |332.12833            |1.8039061127469345           |
|7200.86      |373.07166            |1.608462199664093            |
|7261.91      |393.15918            |1.5392217985817012           |
|7314.675     |319.55624            |1.9075085052182401           |
|7371.05      |403.25418            |1.5232430659788354           |
|7435.16      |333.59668            |1.8573226815923747           |
|7455.07      |357.25583            |1.7389662706455493           |
|7464.64      |391.05334            |1.590712247

**Observations:**
- For most customers, annual_income / 12 is close to the monthly_in-hand_salary (ratio ≈ 1.0), as expected (in-hand salary is after tax)
- A discontinuity between approximately 179,987 and 344,983 in `Annual_Income`. Observations above this breakpoint also showed substantial inconsistency with `Monthly_Inhand_Salary`. 114 rows have unusual high ratio (>2), and 113 of them are above 5 
==>  Values above 180,000 were treated as anomalous for this dataset ==> COnvert to Null


In [114]:
# Clean `Annual_Income`

income_clean_df = income_check_df.withColumn(
    "Annual_Income_clean",
    F.when(
        F.col("Annual_Income") <= 180000,
        F.col("Annual_Income")
    ).otherwise(
        F.lit(None).cast("double")
    )
)

income_clean_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("Annual_Income_clean").isNull().cast("int"))
        .alias("annual_income_nulls")
).show()

+----------+-------------------+
|total_rows|annual_income_nulls|
+----------+-------------------+
|     12500|                114|
+----------+-------------------+



#### 3.6. Cross-check between `Total_EMI_per_month` and `Monthly_inhand_salary`

In [115]:
emi_check_df = income_check_df.withColumn(
    "emi_to_salary_ratio",
    F.when(
        F.col("Monthly_Inhand_Salary") > 0,
        F.col("Total_EMI_per_month") /
        F.col("Monthly_Inhand_Salary")
    )
)

emi_check_df.select(
    "Num_of_Loan_clean",
    "Total_EMI_per_month",
    "Monthly_Inhand_Salary",
    "emi_to_salary_ratio"
).orderBy(
    F.desc("Total_EMI_per_month")
).show(50, truncate=False)

+-----------------+-------------------+---------------------+-------------------+
|Num_of_Loan_clean|Total_EMI_per_month|Monthly_Inhand_Salary|emi_to_salary_ratio|
+-----------------+-------------------+---------------------+-------------------+
|3                |81971.0            |973.9133             |84.16662701744157  |
|1                |81935.0            |8649.47              |9.472835051191383  |
|0                |81760.0            |3186.089             |25.661554697043062 |
|1                |81751.0            |3421.13              |23.895906557278312 |
|3                |81540.0            |6358.4424            |12.82389539620752  |
|2                |81257.0            |4526.4473            |17.95160646564613  |
|9                |81194.0            |4438.3076            |18.293909977211452 |
|0                |81171.0            |2011.5               |40.35346756152125  |
|2                |81125.0            |1454.46              |55.776715879967796 |
|0              

In [116]:
emi_check_df.filter(
    F.col("Num_of_Loan_clean") == 0
).select(
    "Num_of_Loan_clean",
    "Type_of_Loan",
    "Total_EMI_per_month"
).orderBy(
    F.desc("Total_EMI_per_month")
).show(50, truncate=False)

+-----------------+------------+-------------------+
|Num_of_Loan_clean|Type_of_Loan|Total_EMI_per_month|
+-----------------+------------+-------------------+
|0                |NULL        |81760.0            |
|0                |NULL        |81171.0            |
|0                |NULL        |81050.0            |
|0                |NULL        |80983.0            |
|0                |NULL        |78575.0            |
|0                |NULL        |75639.0            |
|0                |NULL        |75532.0            |
|0                |NULL        |70266.0            |
|0                |NULL        |68575.0            |
|0                |NULL        |66949.0            |
|0                |NULL        |66474.0            |
|0                |NULL        |66444.0            |
|0                |NULL        |66045.0            |
|0                |NULL        |64439.0            |
|0                |NULL        |62337.0            |
|0                |NULL        |60579.0       

In [117]:
# Check customer count with EMI > 0 while number_of_loan_clean = 0
emi_check_df.filter(
    (F.col("Num_of_Loan_clean") == 0) &
    (F.col("Total_EMI_per_month") > 0)
).count()

143

In [118]:
emi_exceeds_salary_df = emi_check_df.filter(
    F.col("Total_EMI_per_month") > F.col("Monthly_Inhand_Salary")
)

print(f"Rows where EMI exceeds monthly in-hand salary: {emi_exceeds_salary_df.count()}")

emi_exceeds_salary_df.select(
    "Customer_ID",
    "Total_EMI_per_month",
    "Monthly_Inhand_Salary",
    "Num_of_Loan_clean",
    "snapshot_date"
).show(truncate=False)

Rows where EMI exceeds monthly in-hand salary: 397
+-----------+-------------------+---------------------+-----------------+-------------+
|Customer_ID|Total_EMI_per_month|Monthly_Inhand_Salary|Num_of_Loan_clean|snapshot_date|
+-----------+-------------------+---------------------+-----------------+-------------+
|CUS_0x12fb |57567.0            |1949.2426            |1                |2023-01-01   |
|CUS_0x16f4 |52215.0            |2592.5808            |3                |2023-01-01   |
|CUS_0x2baf |80768.0            |4939.92              |5                |2023-01-01   |
|CUS_0x3433 |7354.0             |1366.2312            |0                |2023-01-01   |
|CUS_0x36b3 |17653.0            |1228.6438            |4                |2023-01-01   |
|CUS_0x3e6d |22409.0            |9270.065             |4                |2023-01-01   |
|CUS_0x48ae |74503.0            |12222.09             |2                |2023-01-01   |
|CUS_0x553e |70266.0            |9276.62              |0             

In [119]:
emi = F.col("Total_EMI_per_month")
salary = F.col("Monthly_Inhand_Salary")

whole_emi_df = emi_check_df.filter(
    (emi > 0) & (emi == F.floor(emi))
)

whole_emi_df.agg(
    F.sum(
        F.when(emi < salary, 1).otherwise(0)
    ).alias("emi_below_salary"),
    F.sum(
        F.when(emi > salary, 1).otherwise(0)
    ).alias("emi_above_salary")
).show()

# Num_of_Loan_clean values for rows where whole-number EMI is below salary
(
    whole_emi_df
    .filter(emi < salary)
    .groupBy("Num_of_Loan_clean")
    .count()
    .orderBy("Num_of_Loan_clean")
    .show()
)

+----------------+----------------+
|emi_below_salary|emi_above_salary|
+----------------+----------------+
|              15|             397|
+----------------+----------------+

+-----------------+-----+
|Num_of_Loan_clean|count|
+-----------------+-----+
|                0|    4|
|                1|    1|
|                2|    5|
|                3|    1|
|                4|    1|
|                6|    1|
|                7|    2|
+-----------------+-----+



**Observations:**
- 143 customers without loans have positive EMI ==> Not plausible
- 397 Monthly EMI larger than the monthly in-hand salary ==> not realistic for a repayment obligation. These EMI values are also in whole number while most other EMI in double format
- 15 rows with EMI as a positive whole number, 4 of which has num_of_loan_clean = 0

==> **Cleaning rules:** 
- EMI < 0 → NULL
- EMI > 0 and Num_of_Loan_clean = 0 → NULL
- EMI is whole number and EMI > Monthly_Inhand_Salary → NULL
- Otherwise → retain


In [120]:
emi = F.col("Total_EMI_per_month")
salary = F.col("Monthly_Inhand_Salary")
num_loans = F.col("Num_of_Loan_clean")

emi_clean_df = (
    emi_check_df
    .withColumn(
        "Total_EMI_per_month_clean",
        F.when(
            emi < 0,
            F.lit(None).cast("double")
        )
        .when(
            (num_loans == 0) & (emi > 0),
            F.lit(None).cast("double")
        )
        .when(
            (emi > 0)
            & (emi == F.floor(emi))
            & (emi > salary),
            F.lit(None).cast("double")
        )
        .otherwise(emi)
    )
)


#### 3.7. Parse `Credit_History_Age` to months

COnvert `Credit_History_Age` from text to number of months: 

E.g: "16 Years and 2 Months" → 16 × 12 + 2 = 194 months

In [121]:
history_years = F.regexp_extract(F.col("Credit_History_Age"), r"(\d+)\s+Years?", 1)
history_months = F.regexp_extract(F.col("Credit_History_Age"), r"(\d+)\s+Months?", 1)

history_clean_df = emi_clean_df.withColumn(
    "credit_history_age_months",
    F.when(
        (history_years == "") | (history_months == ""),
        F.lit(None).cast(IntegerType())
    ).otherwise(
        (history_years.cast(IntegerType()) * 12 + history_months.cast(IntegerType())).cast(IntegerType())
    )
)

history_clean_df.select("Credit_History_Age", "credit_history_age_months").show(5, truncate=False)
history_clean_df.select(
    F.min("credit_history_age_months").alias("min_months"),
    F.max("credit_history_age_months").alias("max_months"),
    F.sum(F.when(F.col("credit_history_age_months").isNull(), 1).otherwise(0)).alias("null_months")
).show()

+---------------------+-------------------------+
|Credit_History_Age   |credit_history_age_months|
+---------------------+-------------------------+
|19 Years and 9 Months|237                      |
|30 Years and 8 Months|368                      |
|15 Years and 9 Months|189                      |
|32 Years and 8 Months|392                      |
|13 Years and 8 Months|164                      |
+---------------------+-------------------------+
only showing top 5 rows

+----------+----------+-----------+
|min_months|max_months|null_months|
+----------+----------+-----------+
|         8|       404|          0|
+----------+----------+-----------+



In [122]:
history_col = F.col("Credit_History_Age")
years = F.regexp_extract(
    history_col,
    r"(\d+)\s+Years?",
    1
).cast("int")

months = F.regexp_extract(
    history_col,
    r"(\d+)\s+Months?",
    1
).cast("int")

history_clean_df = emi_clean_df.withColumn(
    "credit_history_age_months",
    F.when(
        history_col.rlike(
            r"^\s*\d+\s+Years?\s+and\s+\d+\s+Months?\s*$"
        ),
        years * 12 + months
    ).otherwise(
        F.lit(None).cast("int")
    )
)

history_clean_df.select(
    "Credit_History_Age",
    "credit_history_age_months"
).show(10, truncate=False)

+---------------------+-------------------------+
|Credit_History_Age   |credit_history_age_months|
+---------------------+-------------------------+
|19 Years and 9 Months|237                      |
|30 Years and 8 Months|368                      |
|15 Years and 9 Months|189                      |
|32 Years and 8 Months|392                      |
|13 Years and 8 Months|164                      |
|22 Years and 0 Months|264                      |
|9 Years and 9 Months |117                      |
|21 Years and 4 Months|256                      |
|15 Years and 4 Months|184                      |
|33 Years and 5 Months|401                      |
+---------------------+-------------------------+
only showing top 10 rows



#### 3.8. Apply cleaning rules in Range Check (3.2) to the remaining columns

| Column | Valid range in Silver | Reason |
|---|---|---|
| `Num_Bank_Accounts` | 0 – 11 | Cluster ends at 11, next value 26; negative count impossible |
| `Num_Credit_Card` | 0 – 11 | Cluster ends at 11, next value 21 |
| `Interest_Rate` | 0 – 34 (%) | Cluster ends at 34, next value 99 |
| `Num_of_Delayed_Payment` | 0 – 28 | Cluster ends at 28, next value 47; negative count impossible |
| `Num_Credit_Inquiries` | 0 – 17 | Cluster ends at 17, next value 26 |
| `Delay_from_due_date` | Keep as observed | Negative is plausible, in a continuous range |
| `Changed_Credit_Limit` | Keep as observed | Negative is plausible, in a continuous range |

In [123]:
range_rules = {
    "Num_Bank_Accounts": (0, 11),
    "Num_Credit_Card": (0, 11),
    "Interest_Rate": (1, 34),
    "Num_of_Delayed_Payment": (0, 28),
    "Num_Credit_Inquiries": (0, 17)
}

numeric_clean_df = history_clean_df

for column, (lower, upper) in range_rules.items():
    numeric_clean_df = numeric_clean_df.withColumn(
        f"{column}_clean",
        F.when(
            F.col(column).between(lower, upper),
            F.col(column)
        ).otherwise(
            F.lit(None).cast(numeric_clean_df.schema[column].dataType)
        )
    )

for column in range_rules.keys():

    clean_col = f"{column}_clean"

    changed_rows = numeric_clean_df.filter(
        ~F.col(column).eqNullSafe(F.col(clean_col))
    ).count()

    print(f"{column}: {changed_rows} rows cleaned")

Num_Bank_Accounts: 171 rows cleaned
Num_Credit_Card: 296 rows cleaned
Interest_Rate: 270 rows cleaned
Num_of_Delayed_Payment: 189 rows cleaned
Num_Credit_Inquiries: 195 rows cleaned


In [124]:
financials_silver_df = numeric_clean_df.select(
    F.col("Customer_ID").alias("customer_id"),

    F.when(
        F.col("Annual_Income") <= 180000,
        F.col("Annual_Income")
    ).otherwise(
        F.lit(None).cast("double")
    ).alias("annual_income"),

    F.col("Monthly_Inhand_Salary")
        .cast("double")
        .alias("monthly_inhand_salary"),

    F.col("Num_Bank_Accounts_clean")
        .cast("int")
        .alias("num_bank_accounts"),

    F.col("Num_Credit_Card_clean")
        .cast("int")
        .alias("num_credit_card"),

    F.col("Interest_Rate_clean")
        .cast("int")
        .alias("interest_rate"),

    F.col("Num_of_Loan_clean")
        .cast("int")
        .alias("num_of_loan"),

    F.col("Type_of_Loan")
        .alias("type_of_loan"),

    F.col("Delay_from_due_date")
        .cast("int")
        .alias("delay_from_due_date"),

    F.col("Num_of_Delayed_Payment_clean")
        .cast("int")
        .alias("num_of_delayed_payment"),

    F.col("Changed_Credit_Limit")
        .cast("double")
        .alias("changed_credit_limit"),

    F.col("Num_Credit_Inquiries_clean")
        .cast("int")
        .alias("num_credit_inquiries"),

    F.col("Credit_Mix")
        .alias("credit_mix"),

    F.col("Outstanding_Debt")
        .cast("double")
        .alias("outstanding_debt"),

    F.col("Credit_Utilization_Ratio")
        .cast("double")
        .alias("credit_utilization_ratio"),

    F.col("credit_history_age_months")
        .cast("int")
        .alias("credit_history_age_months"),

    F.col("Payment_of_Min_Amount")
        .alias("payment_of_min_amount"),

    F.col("Total_EMI_per_month_clean")
        .cast("double")
        .alias("total_emi_per_month"),

    F.col("Amount_invested_monthly")
        .cast("double")
        .alias("amount_invested_monthly"),

    F.col("Payment_Behaviour")
        .alias("payment_behaviour"),

    F.col("Monthly_Balance")
        .cast("double")
        .alias("monthly_balance"),

    F.to_date("snapshot_date")
        .alias("snapshot_date")
)

## Build Silver Table for `Features_financials`

In [135]:
bronze_directory = "datamart/bronze/financials"
silver_directory = "datamart/silver/financials"

for snapshot_date in date_str_lst:
    utils.data_processing_silver_table.process_feature_financials_silver_table(
        snapshot_date,
        bronze_directory,
        silver_directory,
        spark
    )

loaded from: datamart/bronze/financials/bronze_feature_financials_2023_01_01.csv row count: 530


saved to: datamart/silver/financials/silver_feature_financials_2023_01_01.parquet row count: 530
loaded from: datamart/bronze/financials/bronze_feature_financials_2023_02_01.csv row count: 501
saved to: datamart/silver/financials/silver_feature_financials_2023_02_01.parquet row count: 501
loaded from: datamart/bronze/financials/bronze_feature_financials_2023_03_01.csv row count: 506
saved to: datamart/silver/financials/silver_feature_financials_2023_03_01.parquet row count: 506
loaded from: datamart/bronze/financials/bronze_feature_financials_2023_04_01.csv row count: 510
saved to: datamart/silver/financials/silver_feature_financials_2023_04_01.parquet row count: 510
loaded from: datamart/bronze/financials/bronze_feature_financials_2023_05_01.csv row count: 521
saved to: datamart/silver/financials/silver_feature_financials_2023_05_01.parquet row count: 521
loaded from: datamart/bronze/financials/bronze_feature_financials_2023_06_01.csv row count: 517
saved to: datamart/silver/financial

saved to: datamart/silver/financials/silver_feature_financials_2024_01_01.parquet row count: 485
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_02_01.csv row count: 518


saved to: datamart/silver/financials/silver_feature_financials_2024_02_01.parquet row count: 518
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_03_01.csv row count: 511
saved to: datamart/silver/financials/silver_feature_financials_2024_03_01.parquet row count: 511
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_04_01.csv row count: 513
saved to: datamart/silver/financials/silver_feature_financials_2024_04_01.parquet row count: 513
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_05_01.csv row count: 491
saved to: datamart/silver/financials/silver_feature_financials_2024_05_01.parquet row count: 491
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_06_01.csv row count: 498
saved to: datamart/silver/financials/silver_feature_financials_2024_06_01.parquet row count: 498
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_07_01.csv row count: 505


saved to: datamart/silver/financials/silver_feature_financials_2024_07_01.parquet row count: 505
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_08_01.csv row count: 543
saved to: datamart/silver/financials/silver_feature_financials_2024_08_01.parquet row count: 543
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_09_01.csv row count: 493
saved to: datamart/silver/financials/silver_feature_financials_2024_09_01.parquet row count: 493
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_10_01.csv row count: 456
saved to: datamart/silver/financials/silver_feature_financials_2024_10_01.parquet row count: 456
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_11_01.csv row count: 488
saved to: datamart/silver/financials/silver_feature_financials_2024_11_01.parquet row count: 488
loaded from: datamart/bronze/financials/bronze_feature_financials_2024_12_01.csv row count: 515
saved to: datamart/silver/financial

## EDA & Silver Preparation for `features_clickstream`

### Exploration at file-level

In [ ]:
# Explore data at file-level
clickstream_dir = "datamart/bronze/clickstream"
clickstream_files = sorted(glob.glob(os.path.join(clickstream_dir, "*.csv")))

FEATURE_COLS = [f"fe_{i}" for i in range(1, 21)]
BUSINESS_KEY = ["Customer_ID", "snapshot_date"]

results = []

for file_path in clickstream_files:
    file_name = os.path.basename(file_path)

    # Expected Bronze naming: bronze_feature_clickstream_YYYY_MM_DD.csv
    snapshot_date = file_name.replace("bronze_feature_clickstream_", "").replace(".csv", "")

    # Read Bronze consistently as strings; type casting will be enforced in Silver
    df = spark.read.csv(file_path, header=True, inferSchema=False)

    row_count = df.count()
    column_count = len(df.columns)
    distinct_customer_count = df.select("customer_id").distinct().count()

    results.append({
        "snapshot_date": snapshot_date,
        "row_number": row_count,
        "column_number": column_count,
        "distinct_customer_id": distinct_customer_count,
        "customer_id_unique_within_snapshot": (
            row_count == distinct_customer_count
        )
    })

clickstream_summary = pd.DataFrame(results).sort_values("snapshot_date")
clickstream_summary

,snapshot_date,row_number,column_number,distinct_customer_id,customer_id_unique_within_snapshot
0,2023_01_01,8974,22,8974,True
1,2023_02_01,8974,22,8974,True
2,2023_03_01,8974,22,8974,True
3,2023_04_01,8974,22,8974,True
4,2023_05_01,8974,22,8974,True
5,2023_06_01,8974,22,8974,True
6,2023_07_01,8974,22,8974,True
7,2023_08_01,8974,22,8974,True
8,2023_09_01,8974,22,8974,True
9,2023_10_01,8974,22,8974,True


### Exploration at union-level

In [138]:
# Union all partitions into one dataframe
clickstream_dfs = [
    spark.read.csv(file_path, header=True, inferSchema=False)
    for file_path in clickstream_files
]

clickstream_df = clickstream_dfs[0]
for partition_df in clickstream_dfs[1:]:
    clickstream_df = clickstream_df.unionByName(partition_df)

print(f"Total rows: {clickstream_df.count()}")
clickstream_df.printSchema()
clickstream_df.show(5, truncate=False)

Total rows: 215376
root
 |-- fe_1: string (nullable = true)
 |-- fe_2: string (nullable = true)
 |-- fe_3: string (nullable = true)
 |-- fe_4: string (nullable = true)
 |-- fe_5: string (nullable = true)
 |-- fe_6: string (nullable = true)
 |-- fe_7: string (nullable = true)
 |-- fe_8: string (nullable = true)
 |-- fe_9: string (nullable = true)
 |-- fe_10: string (nullable = true)
 |-- fe_11: string (nullable = true)
 |-- fe_12: string (nullable = true)
 |-- fe_13: string (nullable = true)
 |-- fe_14: string (nullable = true)
 |-- fe_15: string (nullable = true)
 |-- fe_16: string (nullable = true)
 |-- fe_17: string (nullable = true)
 |-- fe_18: string (nullable = true)
 |-- fe_19: string (nullable = true)
 |-- fe_20: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: string (nullable = true)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|f

#### Check duplicates

In [140]:
# Number of exact duplicate rows (excluding the first occurrence)
exact_duplicate_rows = (
    clickstream_df.groupBy(clickstream_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .select(F.sum(F.col("count") - 1).alias("duplicate_rows"))
    .first()["duplicate_rows"] or 0
)

# Number of duplicated (snapshot_date, customer_id) pairs
duplicated_pairs = (
    clickstream_df.groupBy("snapshot_date", "Customer_ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_id = clickstream_df.filter(F.col("Customer_ID").isNull()).count()
null_snapshot_date = clickstream_df.filter(F.col("snapshot_date").isNull()).count()

print(f"Number of exact duplicated rows: {exact_duplicate_rows}")
print(f"Number of duplicated (snapshot_date, customer_id) pairs: {duplicated_pairs}")
print(f"Null Customer_ID: {null_customer_id}")
print(f"Null snapshot_date: {null_snapshot_date}")

Number of exact duplicated rows: 0
Number of duplicated (snapshot_date, customer_id) pairs: 0
Null Customer_ID: 0
Null snapshot_date: 0


#### Check missing, blank

In [141]:
# Values treated as missing, regardless of letter case
missing_values = ("", "unknown", "n/a", "na", "nan", "null", "none")

missing_count_expressions = []

for column_name in clickstream_df.columns:
    column_text = F.lower(
        F.trim(F.col(column_name).cast("string"))
    )
    is_missing = (
        F.col(column_name).isNull()
        | column_text.isin(*missing_values)
    )

    missing_count_expressions.append(
        F.sum(F.when(is_missing, 1).otherwise(0)).alias(column_name)
    )

clickstream_df.select(*missing_count_expressions).show(truncate=False)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|0   |0   |0   |0   |0   |0   |0   |0   |0   |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0          |0            |
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+



### Numeric Parsing check

In [144]:
numeric_parse_summary = []

for column_name in FEATURE_COLS:
    raw_value = F.trim(F.col(column_name).cast("string"))
    parsed_value = F.expr(
        f"try_cast(trim(`{column_name}`) as double)"
    )

    stats = clickstream_df.select(
        F.sum(
            F.when(
                raw_value.isNotNull()
                & (raw_value != "")
                & parsed_value.isNull(),
                1
            ).otherwise(0)
        ).alias("non_numeric_count"),
        F.min(parsed_value).alias("min"),
        F.percentile_approx(parsed_value, 0.01).alias("p01"),
        F.percentile_approx(parsed_value, 0.50).alias("median"),
        F.percentile_approx(parsed_value, 0.99).alias("p99"),
        F.max(parsed_value).alias("max"),
        F.sum(F.when(parsed_value < 0, 1).otherwise(0)).alias("negative_count"),
        F.sum(F.when(parsed_value == 0, 1).otherwise(0)).alias("zero_count"),
        F.countDistinct(parsed_value).alias("n_unique"),
    ).first().asDict()

    numeric_parse_summary.append({"column": column_name, **stats})

numeric_parse_summary = pd.DataFrame(numeric_parse_summary)
numeric_parse_summary

,column,non_numeric_count,min,p01,median,p99,max,negative_count,zero_count,n_unique
0,fe_1,0,-378.0,-130.0,101.0,334.0,541.0,33159,514,763
1,fe_2,0,-356.0,-129.0,103.0,335.0,560.0,32170,528,760
2,fe_3,0,-399.0,-130.0,104.0,338.0,583.0,32079,475,766
3,fe_4,0,-307.0,-129.0,106.0,338.0,562.0,31150,502,764
4,fe_5,0,-343.0,-129.0,107.0,341.0,570.0,30734,479,764
5,fe_6,0,-321.0,-130.0,103.0,337.0,565.0,32353,520,755
6,fe_7,0,-368.0,-126.0,107.0,340.0,537.0,30479,483,778
7,fe_8,0,-361.0,-123.0,111.0,343.0,573.0,28891,485,768
8,fe_9,0,-328.0,-119.0,115.0,347.0,577.0,27224,461,756
9,fe_10,0,-317.0,-116.0,118.0,353.0,537.0,25995,426,773


**Observations:**
- All values parsed cleanly
- Many negative values but features are anonymised without business interpretation ==> Retain original values

In [147]:
clickstream_silver_df = clickstream_df

for column_name in FEATURE_COLS:
    clickstream_silver_df = clickstream_silver_df.withColumn(
        column_name,
        F.expr(f"try_cast(trim(`{column_name}`) as int)")
    )

clickstream_silver_df = (
    clickstream_silver_df
    .withColumn(
        "customer_id",
        F.trim(F.col("Customer_ID").cast("string"))
    )
    .withColumn(
        "snapshot_date",
        F.to_date(F.trim(F.col("snapshot_date")))
    )
    .select(
        *FEATURE_COLS,
        "customer_id",
        "snapshot_date"
    )
)

clickstream_silver_df.printSchema()
clickstream_silver_df.show(5, truncate=False)

root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe

In [148]:
# Correlation among numeric clickstream feature columns
numeric_cols = [c for c in clickstream_silver_df.columns if c.startswith("fe_")]

# Convert to pandas for correlation analysis
feature_corr_df = clickstream_silver_df.select(*numeric_cols).toPandas().corr(method="pearson").round(4)

print("Pairwise correlation matrix:")
display(feature_corr_df)

# Top absolute correlations (excluding self-correlations)
corr_long = (
    feature_corr_df
    .stack()
    .reset_index()
    .rename(columns={"level_0": "col_a", "level_1": "col_b", 0: "corr"})
)

corr_long = corr_long[corr_long["col_a"] != corr_long["col_b"]]
corr_long["abs_corr"] = corr_long["corr"].abs()

top_corr = (
    corr_long
    .sort_values("abs_corr", ascending=False)
    .head(20)
    [["col_a", "col_b", "corr", "abs_corr"]]
)

print("\nTop absolute correlations:")
display(top_corr)

Pairwise correlation matrix:


,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
fe_1,1.0000,-0.0038,-0.0009,0.0017,0.0027,-0.0043,-0.0019,-0.0007,-0.0048,-0.0023,-0.0013,-0.0032,0.0003,-0.0000,-0.0023,0.0018,0.0025,-0.0023,-0.0010,0.0011
fe_2,-0.0038,1.0000,0.0013,0.0045,0.0059,-0.0014,-0.0049,-0.0044,-0.0043,-0.0056,-0.0002,0.0008,-0.0040,0.0033,-0.0014,-0.0035,-0.0002,-0.0004,0.0001,-0.0002
fe_3,-0.0009,0.0013,1.0000,0.0078,0.0064,-0.0037,-0.0056,-0.0097,-0.0056,-0.0082,-0.0030,0.0010,0.0004,-0.0036,-0.0019,0.0026,0.0011,0.0004,0.0015,0.0011
fe_4,0.0017,0.0045,0.0078,1.0000,0.0076,-0.0022,-0.0029,-0.0069,-0.0095,-0.0112,-0.0007,0.0004,-0.0011,-0.0021,-0.0015,-0.0011,-0.0017,-0.0007,0.0010,0.0011
fe_5,0.0027,0.0059,0.0064,0.0076,1.0000,-0.0006,-0.0018,-0.0070,-0.0123,-0.0143,0.0031,0.0006,0.0016,-0.0001,0.0024,0.0010,-0.0010,0.0008,-0.0003,0.0021
fe_6,-0.0043,-0.0014,-0.0037,-0.0022,-0.0006,1.0000,0.0003,0.0010,-0.0013,0.0055,-0.0013,0.0034,-0.0009,-0.0030,0.0013,0.0020,0.0025,-0.0009,0.0046,0.0020
fe_7,-0.0019,-0.0049,-0.0056,-0.0029,-0.0018,0.0003,1.0000,0.0019,0.0020,0.0051,0.0029,-0.0015,0.0001,0.0001,-0.0059,0.0020,0.0048,0.0016,-0.0009,0.0012
fe_8,-0.0007,-0.0044,-0.0097,-0.0069,-0.0070,0.0010,0.0019,1.0000,0.0072,0.0071,0.0024,-0.0014,-0.0027,-0.0050,0.0045,-0.0001,-0.0027,0.0030,-0.0031,0.0009
fe_9,-0.0048,-0.0043,-0.0056,-0.0095,-0.0123,-0.0013,0.0020,0.0072,1.0000,0.0059,0.0008,0.0020,-0.0012,0.0013,-0.0005,0.0009,0.0013,0.0014,0.0017,0.0001
fe_10,-0.0023,-0.0056,-0.0082,-0.0112,-0.0143,0.0055,0.0051,0.0071,0.0059,1.0000,0.0030,-0.0061,-0.0009,-0.0019,0.0032,-0.0016,0.0013,-0.0007,-0.0018,0.0009



Top absolute correlations:


,col_a,col_b,corr,abs_corr
89,fe_5,fe_10,-0.0143,0.0143
184,fe_10,fe_5,-0.0143,0.0143
164,fe_9,fe_5,-0.0123,0.0123
88,fe_5,fe_9,-0.0123,0.0123
69,fe_4,fe_10,-0.0112,0.0112
183,fe_10,fe_4,-0.0112,0.0112
142,fe_8,fe_3,-0.0097,0.0097
47,fe_3,fe_8,-0.0097,0.0097
68,fe_4,fe_9,-0.0095,0.0095
163,fe_9,fe_4,-0.0095,0.0095


**Observations:**
- Features are uncorrelated with each other, no redundant columns to drop

## Build Silver tables for `features_clickstream`

In [153]:
bronze_directory = "datamart/bronze/clickstream"
silver_directory = "datamart/silver/clickstream"

for snapshot_date in date_str_lst:
    utils.data_processing_silver_table.process_feature_clickstream_silver_table(
        snapshot_date,
        bronze_directory,
        silver_directory,
        spark
    )

loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_01_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_01_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_02_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_02_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_03_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_03_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_04_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_04_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_05_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_05_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_06_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_06_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_07_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_07_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_08_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_08_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_09_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_09_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_10_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_10_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_11_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_11_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_12_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_12_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_01_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_01_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_02_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_02_01.parquet
l

saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_05_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_06_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_06_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_07_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_07_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_08_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_08_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_09_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_09_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_10_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_10_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_11_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_11_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2024_12_01.csv row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2024_12_01.parquet
l